## Importing libraries

In [29]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [30]:
warnings.filterwarnings("ignore")

## Getting dataset

In [31]:
from google.colab import drive
drive.mount('/content/drive')

path1 = "/content/drive/MyDrive/Colab Notebooks/dm/practicals/online_retail_II_s1.csv"
path2 = "/content/drive/MyDrive/Colab Notebooks/dm/practicals/online_retail_II_s2.csv"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)

cols_to_keep = ["InvoiceNo","StockCode","Description"]
df1.columns = cols_to_keep
df2.columns = cols_to_keep

retail_df = pd.concat([df1, df2],axis=0)


In [33]:
retail_df

,InvoiceNo,StockCode,Description
0,489434,79323P,PINK CHERRY LIGHTS
1,489434,79323W,WHITE CHERRY LIGHTS
2,489434,22041,"RECORD FRAME 7"" SINGLE SIZE"
3,489434,21232,STRAWBERRY CERAMIC TRINKET BOX
4,489434,22064,PINK DOUGHNUT TRINKET POT
...,...,...,...
541904,581587,22899,CHILDREN'S APRON DOLLY GIRL
541905,581587,23254,CHILDRENS CUTLERY DOLLY GIRL
541906,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE
541907,581587,22138,BAKING SET 9 PIECE RETROSPOT


## Removing null values items

In [34]:
retail_df.isna().sum()

,0
InvoiceNo,0
StockCode,0
Description,4382


In [35]:
retail_df = retail_df.dropna(subset=["Description"])
retail_df

,InvoiceNo,StockCode,Description
0,489434,79323P,PINK CHERRY LIGHTS
1,489434,79323W,WHITE CHERRY LIGHTS
2,489434,22041,"RECORD FRAME 7"" SINGLE SIZE"
3,489434,21232,STRAWBERRY CERAMIC TRINKET BOX
4,489434,22064,PINK DOUGHNUT TRINKET POT
...,...,...,...
541904,581587,22899,CHILDREN'S APRON DOLLY GIRL
541905,581587,23254,CHILDRENS CUTLERY DOLLY GIRL
541906,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE
541907,581587,22138,BAKING SET 9 PIECE RETROSPOT


## Getting dataset info

In [36]:
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1062987 entries, 0 to 541908
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   InvoiceNo    1062987 non-null  object
 1   StockCode    1062987 non-null  object
 2   Description  1062987 non-null  object
dtypes: object(3)
memory usage: 32.4+ MB


In [37]:
retail_df.describe()

,InvoiceNo,StockCode,Description
count,1062987,1062987,1062987
unique,49353,4950,5698
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER
freq,1350,5828,5917


## Getting unique invoices and items

In [38]:
len(retail_df["InvoiceNo"].unique())

49353

In [39]:
len(retail_df["StockCode"].unique())

4950

## Removing items which haven't been purchased more than 250 times (as thier are many unique items)

In [40]:
item_counts = retail_df["StockCode"].value_counts()
retail_df = retail_df[retail_df["StockCode"].isin(item_counts[item_counts>=250].index)]
retail_df

,InvoiceNo,StockCode,Description
0,489434,79323P,PINK CHERRY LIGHTS
1,489434,79323W,WHITE CHERRY LIGHTS
2,489434,22041,"RECORD FRAME 7"" SINGLE SIZE"
3,489434,21232,STRAWBERRY CERAMIC TRINKET BOX
4,489434,22064,PINK DOUGHNUT TRINKET POT
...,...,...,...
541903,581587,22613,PACK OF 20 SPACEBOY NAPKINS
541904,581587,22899,CHILDREN'S APRON DOLLY GIRL
541905,581587,23254,CHILDRENS CUTLERY DOLLY GIRL
541907,581587,22138,BAKING SET 9 PIECE RETROSPOT


## Data cleaning

In [41]:
retail_df["StockCode"] = retail_df["StockCode"].str.replace(r"[^0-9]", "", regex=True)
retail_df = retail_df[~retail_df["InvoiceNo"].str.startswith("C")]
retail_df = retail_df[retail_df["StockCode"].astype(str).str.isnumeric()]

### Getting unique item code description

In [42]:
item_counts = retail_df['StockCode'].value_counts()
item_counts.describe()

,count
count,1202.000000
mean,648.955075
std,545.705294
min,234.000000
25%,324.250000
50%,475.500000
75%,770.750000
max,7944.000000


### Getting unique item name description

In [43]:
item_counts = retail_df['Description'].value_counts()
item_counts.describe()

,count
count,1711.000000
mean,455.899474
std,441.814454
min,1.000000
25%,233.500000
50%,349.000000
75%,586.000000
max,5681.000000


### Since item name and code count is different hence investigating further

In [44]:
retail_df.groupby("StockCode")["Description"].nunique().sort_values(ascending=False).head()

,Description
StockCode,
84997,13
85049,10
20713,9
47566,8
23084,7


### putting name of item for each item code which has max frequency

In [45]:
canonical_desc = (
    retail_df.groupby("StockCode")["Description"]
    .agg(lambda x: x.value_counts().idxmax())
)
retail_df["Description"] = retail_df["StockCode"].map(canonical_desc)

In [46]:
retail_df.groupby("StockCode")["Description"].nunique().sort_values(ascending=False).head()

,Description
StockCode,
85232,1
10002,1
85066,1
85065,1
85064,1


In [47]:
item_counts = retail_df['Description'].value_counts().sort_values(ascending=True)
item_counts

,count
Description,
TRIPLE HOOK ANTIQUE IVORY ROSE,234
PANTRY ROLLING PIN,239
PARISIENNE CURIO CABINET,242
WHITE BEADED GARLAND STRING 20LIGHT,243
ASSORTED TUTTI FRUTTI SMALL PURSE,245
...,...
RED 3 PIECE MINI DOTS CUTLERY SET,3790
REGENCY CAKESTAND 3 TIER,4076
SCANDINAVIAN REDS RIBBONS,4121


In [48]:
item_counts = retail_df['StockCode'].value_counts().sort_values()
item_counts

,count
StockCode,
23146,234
22978,239
23112,242
85047,243
21648,245
...,...
84997,3790
22423,4076
85049,4121


### Combining item codes if they have common item name

In [49]:
desc_to_codes = retail_df.groupby("Description")["StockCode"].nunique()
shared = desc_to_codes[desc_to_codes > 1]
print("Descriptions shared by multiple StockCodes:")
print(shared)

Descriptions shared by multiple StockCodes:
Description
COLOURING PENCILS BROWN TUBE    2
Name: StockCode, dtype: int64


In [50]:
for desc in shared.index:
    codes = retail_df[retail_df["Description"] == desc]["StockCode"].unique()
    print(f"\nDescription: '{desc}' → StockCodes: {codes}")
    subset = retail_df[retail_df["Description"] == "COLOURING PENCILS BROWN TUBE"]
    dominant_code = subset["StockCode"].value_counts().idxmax()
    retail_df.loc[retail_df["Description"] == "COLOURING PENCILS BROWN TUBE", "StockCode"] = dominant_code

print("Unique StockCodes:", retail_df["StockCode"].nunique())
print("Unique Descriptions:", retail_df["Description"].nunique())


Description: 'COLOURING PENCILS BROWN TUBE' → StockCodes: ['10133' '10135']
Unique StockCodes: 1201
Unique Descriptions: 1201


In [51]:
retail_df.value_counts()

InvoiceNo  StockCode  Description                     
555524     22698      PINK REGENCY TEACUP AND SAUCER      20
536857     85049      SCANDINAVIAN REDS RIBBONS           14
522760     85049      SCANDINAVIAN REDS RIBBONS           13
555524     22697      GREEN REGENCY TEACUP AND SAUCER     12
536747     85049      SCANDINAVIAN REDS RIBBONS           12
                                                          ..
524726     22535      MAGIC DRAWING SLATE BUNNIES          1
           22536      MAGIC DRAWING SLATE PURDEY           1
           22550      HOLIDAY FUN LUDO                     1
           22557      PLASTERS IN TIN VINTAGE PAISLEY      1
           22444      GROW YOUR OWN PLANT IN A CAN         1
Name: count, Length: 731772, dtype: int64

In [52]:
retail_df

,InvoiceNo,StockCode,Description
0,489434,79323,PINK CHERRY LIGHTS
1,489434,79323,PINK CHERRY LIGHTS
2,489434,22041,"RECORD FRAME 7"" SINGLE SIZE"
3,489434,21232,STRAWBERRY CERAMIC TRINKET BOX
4,489434,22064,PINK DOUGHNUT TRINKET POT
...,...,...,...
541902,581587,23256,CHILDRENS CUTLERY SPACEBOY
541903,581587,22613,PACK OF 20 SPACEBOY NAPKINS
541904,581587,22899,CHILDREN'S APRON DOLLY GIRL
541905,581587,23254,CHILDRENS CUTLERY DOLLY GIRL


## Converting the dataframe to one-hot format

In [53]:
retail_items_df = (
    pd.get_dummies(retail_df["StockCode"])
    .groupby(retail_df["InvoiceNo"])
    .max()  # max ensures binary 0/1
)
retail_items_df

,10002,10135,11001,15034,15036,15039,15044,15056,15060,16156,...,85174,85175,85176,85178,85184,85199,85206,85227,85231,85232
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
489434,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489435,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489436,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489437,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
489438,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581583,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
581584,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
581585,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Importing libraries for fp-growth and rule mining

In [54]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
warnings.filterwarnings(
    "ignore",
    message="datetime.datetime.utcnow() is deprecated"
)
warnings.filterwarnings(
    "ignore",
    message="jupyter_client.session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC)."
)
warnings.filterwarnings(
    "ignore",
    message="datetime.datetime.utcnow() is deprecated"
)
warnings.filterwarnings(
    "ignore",
    message="jupyter_client.session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC)."
)


#### Metrics

In [55]:
min_sup = 0.01
min_conf = 0.6
min_lift = 1.2

### getting frequent itemsets

In [56]:
frequent_itemsets = fpgrowth(retail_items_df, min_support=min_sup, use_colnames=True)

In [57]:
frequent_itemsets = frequent_itemsets.sort_values("support",ascending=False)
frequent_itemsets

,support,itemsets
50,0.138880,(85123)
69,0.135830,(85099)
501,0.101582,(22423)
95,0.080593,(21212)
118,0.078991,(47566)
...,...,...
1232,0.010003,"(85099, 84692)"
1191,0.010003,"(85099, 21843)"
1015,0.010003,"(22423, 21931)"
1290,0.010003,"(22384, 85099, 21931)"


### getting rules

In [58]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_conf)

### getting non trivial rules (removing weak rules)

In [59]:
rules = rules[
    (rules['confidence'] >= min_conf) &
    (rules['lift'] > min_lift) &
    (rules['support'] >= min_sup)
]
rules = rules.sort_values(
    ["lift", "confidence", "support"],
    ascending=False
)

### frequent itemsets

In [60]:
frequent_itemsets

,support,itemsets
50,0.138880,(85123)
69,0.135830,(85099)
501,0.101582,(22423)
95,0.080593,(21212)
118,0.078991,(47566)
...,...,...
1232,0.010003,"(85099, 84692)"
1191,0.010003,"(85099, 21843)"
1015,0.010003,"(22423, 21931)"
1290,0.010003,"(22384, 85099, 21931)"


### rules

In [61]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
213,"(22745, 22748)",(22746),0.014397,0.014578,0.010572,0.734291,50.369227,1.0,0.010362,3.708648,0.994464,0.574438,0.730360,0.729734
214,(22746),"(22745, 22748)",0.014578,0.014397,0.010572,0.725177,50.369227,1.0,0.010362,3.586322,0.994647,0.574438,0.721163,0.729734
212,"(22748, 22746)",(22745),0.012252,0.018171,0.010572,0.862869,47.486036,1.0,0.010349,7.159799,0.991084,0.532552,0.860331,0.722331
211,"(22745, 22746)",(22748),0.011916,0.019179,0.010572,0.887202,46.258842,1.0,0.010343,8.695355,0.990181,0.515113,0.884996,0.719207
158,(22746),(22745),0.014578,0.018171,0.011916,0.817376,44.982416,1.0,0.011651,5.376229,0.992234,0.571960,0.813996,0.736568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,(22804),(85123),0.017344,0.138880,0.012252,0.706408,5.086456,1.0,0.009843,2.933052,0.817579,0.085099,0.659058,0.397314
3,(21733),(85123),0.045518,0.138880,0.031896,0.700738,5.045628,1.0,0.025575,2.877480,0.840046,0.209153,0.652474,0.465203
2,(22411),(85099),0.056684,0.135830,0.038255,0.674875,4.968515,1.0,0.030555,2.657958,0.846729,0.247989,0.623771,0.478256
32,(23199),(85099),0.025279,0.135830,0.017034,0.673824,4.960782,1.0,0.013600,2.649398,0.819126,0.118227,0.622556,0.399614


### rules generated

In [62]:
print(rules.shape[0],"rules mined")

245 rules mined


### printing all rules

In [63]:
code_to_desc = (
    retail_df[['StockCode', 'Description']]
    .drop_duplicates()
    .set_index('StockCode')['Description']
    .to_dict()
)
def decode_items(itemset):
    return [code_to_desc.get(i, i) for i in itemset]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


rules_df = pd.DataFrame(rules[["antecedents", "consequents"]])
print("total rules generated : ",len(rules_df["antecedents"].to_list()))
rules_df['antecedents'] = rules_df['antecedents'].apply(lambda x : (" , ".join(decode_items(x))).lower())
rules_df['consequents'] = rules_df['consequents'].apply(lambda x : " , ".join(decode_items(x)).lower())
rules_df.index = [x+1 for x in range(len(rules_df["antecedents"].to_list()))]


total rules generated :  245


In [64]:
rules_df

,antecedents,consequents
1,"poppy's playhouse bedroom , poppy's playhouse kitchen",poppy's playhouse livingroom
2,poppy's playhouse livingroom,"poppy's playhouse bedroom , poppy's playhouse kitchen"
3,"poppy's playhouse kitchen , poppy's playhouse livingroom",poppy's playhouse bedroom
4,"poppy's playhouse bedroom , poppy's playhouse livingroom",poppy's playhouse kitchen
5,poppy's playhouse livingroom,poppy's playhouse bedroom
6,poppy's playhouse bedroom,poppy's playhouse livingroom
7,coffee mug dog + ball design,coffee mug cat + bird design
8,coffee mug cat + bird design,coffee mug dog + ball design
9,poppy's playhouse livingroom,poppy's playhouse kitchen
10,poppy's playhouse kitchen,poppy's playhouse livingroom
